In [1]:
import pandas as pd
import dask.dataframe as dd
import json
import matplotlib.pyplot as plt
from scipy import stats

with open("../data/ndc/ARB.json", "r") as f:
    ARBs = json.load(f)

with open("../data/ndc/ACE.json", "r") as f:
    ACEs = json.load(f)

## Medicare Prescription Data
The **Centers for Medicare & Medicaid Services** (CMS) publishes a yearly dataset called the [Medicare Part D Prescribers](https://data.cms.gov/provider-summary-by-type-of-service/medicare-part-d-prescribers/medicare-part-d-prescribers-by-provider-and-drug) data. The dataset contains an aggregation of all the prescriptions given to Medicare Part D beneficiaries by physicians each year.

First we need to understand the difference between the number of prescriptions issued for sacubitril/valsartan (brand name Entresto) versus the number of prescriptions filed for all other drugs. We're going to create a dataset that has one column for Entresto and one column for all other prescriptions.

In [2]:
PART_D_COLS = ["Prscrbr_NPI", "Tot_Clms", "Brnd_Name", "Gnrc_Name"]

PART_D_2015 = dd.read_csv('../data/part-d/2015/2015.csv', dtype='str', usecols=PART_D_COLS, encoding='latin-1')
PART_D_2016 = dd.read_csv('../data/part-d/2016/2016.csv', dtype='str', usecols=PART_D_COLS, encoding='latin-1')
PART_D_2017 = dd.read_csv('../data/part-d/2017/2017.csv', dtype='str', usecols=PART_D_COLS, encoding='latin-1')
PART_D_2018 = dd.read_csv('../data/part-d/2018/2018.csv', dtype='str', usecols=PART_D_COLS, encoding='latin-1')
PART_D_2019 = dd.read_csv('../data/part-d/2019/2019.csv', dtype='str', usecols=PART_D_COLS, encoding='latin-1')
PART_D_2020 = dd.read_csv('../data/part-d/2020/2020.csv', dtype='str', usecols=PART_D_COLS, encoding='latin-1')
PART_D_2021 = dd.read_csv('../data/part-d/2021/2021.csv', dtype='str', usecols=PART_D_COLS, encoding='latin-1')

PART_D_DATASETS = {
    "PART_D_2015" : PART_D_2015,
    "PART_D_2016" : PART_D_2016,
    "PART_D_2017" : PART_D_2017,
    "PART_D_2018" : PART_D_2018,
    "PART_D_2019" : PART_D_2019,
    "PART_D_2020" : PART_D_2020,
    "PART_D_2021" : PART_D_2021
    
}

for name, df in PART_D_DATASETS.items():
    new_df = df.rename(columns={'Prscrbr_NPI':'Covered_Recipient_NPI'})

    # drop rows missing an NPI
    new_df = new_df.dropna(subset=['Covered_Recipient_NPI'])

    # set the index to NPI
    new_df = new_df.set_index('Covered_Recipient_NPI')

    # convert generic names to lowercase for match purposes
    new_df['Gnrc_Name'] = new_df['Gnrc_Name'].str.lower()
    new_df['Year'] = name.split('_')[-1]
    
    PART_D_DATASETS[name] = new_df.compute()

In [3]:
PART_D_DATASET_COMBINED = pd.concat([d[1] for d in PART_D_DATASETS.items()], axis=0, join='outer').reset_index()

In [4]:
assert sum([d[1].shape[0] for d in PART_D_DATASETS.items()]) == PART_D_DATASET_COMBINED.shape[0]

In [5]:
# Get a list of all generic names for ARBs and ACEis
ARB_and_ACE_names = list(ACEs.keys()) + list(ARBs.keys())

# Find all prescriptions for Entresto
ENTRESTO_PRESCRIPTIONS = PART_D_DATASET_COMBINED[PART_D_DATASET_COMBINED['Gnrc_Name'] == 'sacubitril/valsartan'].copy()

# Find all other prescriptions
OTHER_PRESCRIPTIONS = PART_D_DATASET_COMBINED[(PART_D_DATASET_COMBINED['Gnrc_Name'] != 'sacubitril/valsartan') & (PART_D_DATASET_COMBINED['Gnrc_Name'].isin(set(list(ARB_and_ACE_names))))].copy()

# Convert the total prescriptions to number
ENTRESTO_PRESCRIPTIONS.loc[:, 'Total_Clms'] = pd.to_numeric(ENTRESTO_PRESCRIPTIONS['Tot_Clms'])
OTHER_PRESCRIPTIONS.loc[:, 'Total_Clms'] = pd.to_numeric(OTHER_PRESCRIPTIONS['Tot_Clms'])

# Group DFs by their index (physician NPI) to get a sum of all claims per NPI
TOTAL_ENTRESTO_PRESCRIPTIONS_BY_NPI = ENTRESTO_PRESCRIPTIONS.groupby(['Covered_Recipient_NPI', 'Year'])['Total_Clms'].sum()
TOTAL_OTHER_PRESCRIPTIONS_BY_NPI = OTHER_PRESCRIPTIONS.groupby(['Covered_Recipient_NPI', 'Year'])['Total_Clms'].sum()

In [6]:
# Create a merged DF that has a column for Entresto prescriptions and a column for other prescriptions, per physician
MERGED_PRESCRIPTION_DF = pd.merge(TOTAL_OTHER_PRESCRIPTIONS_BY_NPI, TOTAL_ENTRESTO_PRESCRIPTIONS_BY_NPI, on=['Covered_Recipient_NPI', 'Year'], how='outer')
MERGED_PRESCRIPTION_DF.rename(columns={'Total_Clms_x':'OTHER_PRESCRIPTIONS', 'Total_Clms_y':'ENTRESTO_PRESCRIPTIONS'}, inplace=True)

In [43]:
MERGED_PRESCRIPTION_DF.sort_values(by='OTHER_PRESCRIPTIONS', ascending=False).fillna(0).head()

OTHER_PRESCRIPTIONS  ENTRESTO_PRESCRIPTIONS
Covered_Recipient_NPI YEAR                                             
1285761833            2018                22289                      51
                      2019                19066                      63
                      2017                17914                       0
                      2020                12469                      81
                      2016                11471                       0

We now have a DataFrame (`MERGED_PRESCRIPTION_DF`) that has four columns:
- `Covered_Recipient_NPI` (index): the physician's NPI number
- `Year`: the year the prescriptions were written
- `OTHER_PRESCRIPTIONS`: the number of prescriptions the physician wrote that were not for Entresto
- `ENTRESTO_PRESCRIPTIONS`: the number of prescriptions the physician wrote for Entresto

## Payments to Doctors data
Next we need to find out how many payments each physician received for each year. The CMS also publishes a yearly dataset called [Open Payments](https://www.cms.gov/priorities/key-initiatives/open-payments/law-policy) which is a federally mandated report that discloses all payments made to physicians by entities like pharmaceutical companies. 

For the physicians in our prescription dataset, we need to compile a corresponding dataset that contains the total number of payments that physician received. This dataset needs to separate payments into two categories:
1) Payments that relate to the drug Entresto
2) Payments that do *not* pertain to Entresto

We need the data separated this way so we can examine the correlation between the number of payments a doctor received related to Entresto and the number of prescriptions for Entresto they made that year; it's important to note that we need **both** payments for Entresto and payments received for anything else besides Entresto. If we only include doctors who received _payments_ for Entresto in our analysis, we'll be excluding any doctors who have _not_ received payments for Entresto but may have still given out prescriptions.

In [8]:
'''
These columns represent the relevant drug or therapeutic in which the payment is being made for the 2015 year. 
These columns were renamed after 2015.
'''
COVERED_DRUG_COLUMNS_2015 = [
    'NDC_of_Associated_Covered_Drug_or_Biological1',
    'NDC_of_Associated_Covered_Drug_or_Biological2',
    'NDC_of_Associated_Covered_Drug_or_Biological3',
    'NDC_of_Associated_Covered_Drug_or_Biological4',
    'NDC_of_Associated_Covered_Drug_or_Biological5',
]
NPI_COLUMN_2015 = ["Physician_NPI"]

'''
These are the current columns that represent the covered drug.
'''
COVERED_DRUG_COLUMNS_CURRENT = [
    "Associated_Drug_or_Biological_NDC_1",
    "Associated_Drug_or_Biological_NDC_2",
    "Associated_Drug_or_Biological_NDC_3",
    "Associated_Drug_or_Biological_NDC_4",
    "Associated_Drug_or_Biological_NDC_5",
]
NPI_COLUMN_CURRENT = ["Covered_Recipient_NPI"]

PAYMENT_TYPE_COLUMN = ["Nature_of_Payment_or_Transfer_of_Value"]
PAYMENT_DATE_COLUMN = ["Date_of_Payment"]
RECORD_ID_COLUMN = ["Record_ID"]
UNIVERSAL_COLUMNS = PAYMENT_TYPE_COLUMN + PAYMENT_DATE_COLUMN + RECORD_ID_COLUMN

COLUMNS_2015 = COVERED_DRUG_COLUMNS_2015 + NPI_COLUMN_2015 + UNIVERSAL_COLUMNS
COLUMNS_CURRENT = COVERED_DRUG_COLUMNS_CURRENT + NPI_COLUMN_CURRENT + UNIVERSAL_COLUMNS

# For time and memory constraints, use only 2016 as a model
OPEN_PAYMENTS_2015 = dd.read_csv('../data/open-payments/2015.csv', dtype='str', usecols=COLUMNS_2015)
OPEN_PAYMENTS_2016 = dd.read_csv('../data/open-payments/2016.csv', dtype='str', usecols=COLUMNS_CURRENT)
OPEN_PAYMENTS_2017 = dd.read_csv('../data/open-payments/2017.csv', dtype='str', usecols=COLUMNS_CURRENT)
OPEN_PAYMENTS_2018 = dd.read_csv('../data/open-payments/2018.csv', dtype='str', usecols=COLUMNS_CURRENT)
OPEN_PAYMENTS_2019 = dd.read_csv('../data/open-payments/2019.csv', dtype='str', usecols=COLUMNS_CURRENT)
OPEN_PAYMENTS_2020 = dd.read_csv('../data/open-payments/2020.csv', dtype='str', usecols=COLUMNS_CURRENT)
OPEN_PAYMENTS_2021 = dd.read_csv('../data/open-payments/2021.csv', dtype='str', usecols=COLUMNS_CURRENT)

In [9]:
renamed_columns = dict(zip(COLUMNS_2015, COLUMNS_CURRENT))
OPEN_PAYMENTS_2015 = OPEN_PAYMENTS_2015.rename(columns=renamed_columns)

In [11]:
datasets = {
    "OPEN_PAYMENTS_2015": OPEN_PAYMENTS_2015,
    "OPEN_PAYMENTS_2016": OPEN_PAYMENTS_2016,
    "OPEN_PAYMENTS_2017": OPEN_PAYMENTS_2017,
    "OPEN_PAYMENTS_2018": OPEN_PAYMENTS_2018,
    "OPEN_PAYMENTS_2019": OPEN_PAYMENTS_2019,
    "OPEN_PAYMENTS_2020": OPEN_PAYMENTS_2020,
    "OPEN_PAYMENTS_2021": OPEN_PAYMENTS_2021,
}

'''
Here we are 
1) dropping any rows that are missing an associated drug
2) dropping any rows that are missing a physicians NPI
3) filtering by the product category of cardiovascular
4) filtering for payments of the food and beverage type
5) setting the index column to the NPI
'''
for name, df in datasets.items():
    new_df = df.dropna(subset=['Associated_Drug_or_Biological_NDC_1',
                              'Associated_Drug_or_Biological_NDC_2',
                              'Associated_Drug_or_Biological_NDC_3',
                              'Associated_Drug_or_Biological_NDC_4',
                              'Associated_Drug_or_Biological_NDC_5',],
                                how='all')
    new_df = new_df.dropna(subset=['Covered_Recipient_NPI'])
    new_df = new_df[new_df['Nature_of_Payment_or_Transfer_of_Value'] == 'Food and Beverage']
    new_df['YEAR'] = new_df['Date_of_Payment'].str.split('/').str[-1]

    datasets[name] = new_df.compute()

In [12]:
OPEN_PAYMENTS_DATA = pd.concat([d[1] for d in datasets.items()], axis=0, join='inner').reset_index()

In [13]:
OPEN_PAYMENTS_DATA.head()

,index,Covered_Recipient_NPI,Date_of_Payment,Nature_of_Payment_or_Transfer_of_Value,Record_ID,Associated_Drug_or_Biological_NDC_1,Associated_Drug_or_Biological_NDC_2,Associated_Drug_or_Biological_NDC_3,Associated_Drug_or_Biological_NDC_4,Associated_Drug_or_Biological_NDC_5,YEAR
0,542,1588642383,02/12/2015,Food and Beverage,230591548,24987-111-14,<NA>,<NA>,<NA>,<NA>,2015
1,543,1790747293,03/26/2015,Food and Beverage,230591550,24987-111-14,<NA>,<NA>,<NA>,<NA>,2015
2,544,1790747293,02/18/2015,Food and Beverage,230591552,24987-111-14,<NA>,<NA>,<NA>,<NA>,2015
3,545,1265596969,02/19/2015,Food and Beverage,230591554,24987-111-14,<NA>,<NA>,<NA>,<NA>,2015
4,546,1265596969,03/27/2015,Food and Beverage,230591556,24987-111-14,<NA>,<NA>,<NA>,<NA>,2015


In [14]:
assert sum([d[1].shape[0] for d in datasets.items()]) == OPEN_PAYMENTS_DATA.shape[0]

The Open Payments dataset has a column called `Associated_Drug_or_Biological_NDC_N` which lists the [National Drug Code](https://www.fda.gov/drugs/drug-approvals-and-databases/national-drug-code-directory) that is relevant to the particular payment. So in order to differentiate between payments related to Entresto and all other payments, we need to know the NDC codes for Entresto (reminder that the generic name is "sacubitril and valsartan"). 

The FDA [published](https://open.fda.gov/apis/drug/ndc/) both a static dataset and API for all NDC codes. The FDA also classifies drugs a number of ways, one of which is by their [Established Pharmacologic Class](https://www.fda.gov/industry/structured-product-labeling-resources/pharmacologic-class). Sacubitril/valsartan is considered an angiotensin II receptor blockers (ARB); a similar EPC is angiotensin converting enzyme inhibitors (ACE).

What we need to do is compile a list of all NDC codes that correspond to sacubitril/valsartan as well as a list of all NDCs for all other ARBs and ACEs. Once we have these lists, we can filter the payment data for only the payments that are relevant to this subset of drugs.

In [15]:
# All NCD codes for the ARBs in Entresto class
ENTRESTO_NCDs = ARBs['sacubitril and valsartan']

# All NCD codes for all ARBs that arent Entresto
OTHER_ARB_NCDs = [item for sublist in [items for key, items in ARBs.items() if key != 'sacubitril and valsartan'] for item in sublist]

# All ACEis NCD codes
ACEis = [item for sublist in [item for key, item in ACEs.items()] for item in sublist]

OTHER_NCDS = OTHER_ARB_NCDs + ACEis

# Compute the total number of payments for any ARB or ACE
ALL_PAYMENTS = OPEN_PAYMENTS_DATA[OPEN_PAYMENTS_DATA[[
    'Associated_Drug_or_Biological_NDC_1', 
    'Associated_Drug_or_Biological_NDC_2',
    'Associated_Drug_or_Biological_NDC_3',
    'Associated_Drug_or_Biological_NDC_4',
    'Associated_Drug_or_Biological_NDC_5',
]].isin(ENTRESTO_NCDs + OTHER_NCDS).any(axis=1)]

# Find payments that reference an Entresto NCD
ENTRESTO_PAYMENTS = ALL_PAYMENTS[ALL_PAYMENTS[[
    'Associated_Drug_or_Biological_NDC_1', 
    'Associated_Drug_or_Biological_NDC_2',
    'Associated_Drug_or_Biological_NDC_3',
    'Associated_Drug_or_Biological_NDC_4',
    'Associated_Drug_or_Biological_NDC_5',
]].isin(ENTRESTO_NCDs).any(axis=1)]

ENTRESTO_PAYMENTS.set_index(NPI_COLUMN_CURRENT)

# Find payments that reference any other NCD
OTHER_PAYMENTS = ALL_PAYMENTS[ALL_PAYMENTS[[
    'Associated_Drug_or_Biological_NDC_1', 
    'Associated_Drug_or_Biological_NDC_2',
    'Associated_Drug_or_Biological_NDC_3',
    'Associated_Drug_or_Biological_NDC_4',
    'Associated_Drug_or_Biological_NDC_5',
]].isin(OTHER_NCDS).any(axis=1)]

OTHER_PAYMENTS.set_index(NPI_COLUMN_CURRENT)

# Deduplicate Entresto NCDs
OTHER_PAYMENTS = ALL_PAYMENTS[~ALL_PAYMENTS[[
    'Associated_Drug_or_Biological_NDC_1', 
    'Associated_Drug_or_Biological_NDC_2',
    'Associated_Drug_or_Biological_NDC_3',
    'Associated_Drug_or_Biological_NDC_4',
    'Associated_Drug_or_Biological_NDC_5',
]].isin(ENTRESTO_NCDs).any(axis=1)]

In [16]:
ENTRESTO_PAYMENTS_GROUPED = ENTRESTO_PAYMENTS.groupby(['Covered_Recipient_NPI', 'YEAR']).size().reset_index(['Covered_Recipient_NPI', 'YEAR'])
OTHER_PAYMENTS_GROUPED = OTHER_PAYMENTS.groupby(['Covered_Recipient_NPI', 'YEAR']).size().reset_index(['Covered_Recipient_NPI', 'YEAR'])

# Merge payment data for Entrestro and payment data for other drugs
# Grouped by NPI and year of payment
MERGED_PAYMENT_DATA = pd.merge(
    ENTRESTO_PAYMENTS_GROUPED,
    OTHER_PAYMENTS_GROUPED,
    on=["Covered_Recipient_NPI", "YEAR"],
    how="outer"  # or 'outer' if you want all combinations
).rename(columns={'0_x':'ENTRESTO_PAYMENTS','0_y':'OTHER_PAYMENTS'}).fillna(0)

# Check that all unique NPIs in the other payments filtered dataset match the NPIs in the combined dataset
assert OTHER_PAYMENTS['Covered_Recipient_NPI'].nunique() == MERGED_PAYMENT_DATA[MERGED_PAYMENT_DATA['OTHER_PAYMENTS'] > 0]['Covered_Recipient_NPI'].nunique()
# Same with NPIs for Entresto payments
assert ENTRESTO_PAYMENTS['Covered_Recipient_NPI'].nunique() == MERGED_PAYMENT_DATA[MERGED_PAYMENT_DATA['ENTRESTO_PAYMENTS'] > 0]['Covered_Recipient_NPI'].nunique()

In [17]:
ENTRESTO_CANARY = MERGED_PAYMENT_DATA[
    (MERGED_PAYMENT_DATA['Covered_Recipient_NPI'] == '1023089778')
    & (MERGED_PAYMENT_DATA['YEAR'] == '2018')
]['ENTRESTO_PAYMENTS'].values[0]

OPEN_PAYMENTS_CANARY = OPEN_PAYMENTS_DATA[
    (OPEN_PAYMENTS_DATA['Covered_Recipient_NPI'] == '1023089778') & 
    (OPEN_PAYMENTS_DATA['Associated_Drug_or_Biological_NDC_1'] == '0078-0659-20') & 
    (OPEN_PAYMENTS_DATA['YEAR'] == '2018')].index.size

In [18]:
assert ENTRESTO_CANARY == OPEN_PAYMENTS_CANARY

In [44]:
MERGED_PAYMENT_DATA.sort_values(by='ENTRESTO_PAYMENTS', ascending=False).head()

,Covered_Recipient_NPI,YEAR,ENTRESTO_PAYMENTS,OTHER_PAYMENTS
5744,1023089778,2018,78.0,0.0
42315,1184694028,2018,77.0,0.0
42316,1184694028,2019,76.0,0.0
204884,1891765269,2018,75.0,0.0
191532,1831197268,2018,75.0,0.0


In [20]:
MERGED_PRESCRIPTION_DF.index.names = ['Covered_Recipient_NPI', 'YEAR']
MERGED_DATASET = pd.merge(MERGED_PRESCRIPTION_DF, MERGED_PAYMENT_DATA, on=["Covered_Recipient_NPI", "YEAR"], how='outer').fillna(0)
MERGED_DATASET.sort_values(by='OTHER_PAYMENTS', ascending=False).head()

,Covered_Recipient_NPI,YEAR,OTHER_PRESCRIPTIONS,ENTRESTO_PRESCRIPTIONS,ENTRESTO_PAYMENTS,OTHER_PAYMENTS
1361920,1629048012,2016,290,0,11.0,79.0
241907,1114033040,2016,423,0,0.0,66.0
921350,1427012574,2016,1114,0,19.0,59.0
865822,1396740536,2016,284,0,0.0,52.0
772346,1356346597,2016,382,0,0.0,52.0


## Data Integrity
Now we have a dataset that combines payment data with prescription data that is ready to be analyzed. We should do some data integrity checks to make sure our transformations were applied properly:

1) The first thing we did was take the total number of physicians included in the Open Payments data and filtered out only the payments related to food and beverage expenses: $48,988,175$

In [21]:
f'{len(OPEN_PAYMENTS_DATA):,}'

'48,988,175'

2) Then we filtered the Open Payment data for payments made for Entresto versus payments made for other heart drugs: 443,712 payments were made in reference to Entresto while 32,564 were for other heart drugs. We also computed that 476,276 total payments were made across all NCDs in the ARB/ACE class.

Therefore, for confirmation: $806,266 + 48,038 = 854,304$ 

In [22]:
f'{len(ENTRESTO_PAYMENTS):,} + {len(OTHER_PAYMENTS):,} = {len(ALL_PAYMENTS):,}'

'806,266 + 48,038 = 854,304'

In [23]:
assert len(ENTRESTO_PAYMENTS) + len(OTHER_PAYMENTS) == len(ALL_PAYMENTS)

3) Next we filtered our prescription data; between 2015 and 2021, there were a total of $175,853,385$ prescriptions filed under Medicare Part D; $6,240,844$ that referenced any of the generic names for the drugs in our ARB and ACE categories; $114,166$ were for Entresto.

In [24]:
f'{len(PART_D_DATASET_COMBINED):,} | {len(OTHER_PRESCRIPTIONS):,} | {len(ENTRESTO_PRESCRIPTIONS):,}'

'175,853,385 | 6,240,844 | 114,166'

4) There are $92,686$ doctors in `ENTRESTO_PAYMENTS` and $17,209$ doctors in `OTHER_PAYMENTS`; many more doctors got paid by the manufacturer for Entresto than by any other drug company.

In [42]:
f"{ENTRESTO_PAYMENTS['Covered_Recipient_NPI'].nunique():,} | {OTHER_PAYMENTS['Covered_Recipient_NPI'].nunique():,}"

'92,686 | 17,209'

# Analysis

We're ready to perform some analysis on the data; first, let's compute the odds ratio for physicians who received at least 10 payments versus the physicians who did not. To do this, we need a 2x2 matrix with the following cohorts:
<table>
  <tr>
    <th></th>
    <th>Filed Entresto prescriptions</th>
    <th>Did NOT file Entresto prescriptions</th>
        
  </tr>
  <tr>
    <td><b>Received payment</b></td>
    <td>Group A</td>
      <td>Group B</td>
  </tr>
    <tr>
    <td><b>Did NOT receive payment</b></td>
    <td>Group C</td>
        <td>Group D</td>
  </tr>
</table>

We'll then compute the odds ratio according to
$$
\text{OR} = \frac{A \div B}{C \div D}
$$
where $A \div B$ defines the odds of finding a physician whose received an Entresto payment and prescribed Entresto and $C \div D$ defines the odds of finding a physician who _hasn't_ been paid but still recorded Entresto claims.

In [26]:
MERGED_DATASET.loc[:, "ALL_PRESCRIPTIONS"] = MERGED_DATASET["ENTRESTO_PRESCRIPTIONS"] + MERGED_DATASET["OTHER_PRESCRIPTIONS"]

In [27]:
#A: physicians who made at least 1+ Entresto prescriptions a year and received Entresto payments
A = MERGED_DATASET[
    (MERGED_DATASET["ALL_PRESCRIPTIONS"] >= 70) & 
    (MERGED_DATASET["ENTRESTO_PRESCRIPTIONS"] > 0) &
    (MERGED_DATASET["ENTRESTO_PAYMENTS"] > 0)
    ]

#B: physicians who did not make any Entresto prescriptions and received Entresto payments
B = MERGED_DATASET[
    (MERGED_DATASET["ALL_PRESCRIPTIONS"] >= 70) & 
    (MERGED_DATASET["ENTRESTO_PRESCRIPTIONS"] == 0) &
    (MERGED_DATASET["ENTRESTO_PAYMENTS"] > 0) 
    ]

#C: physicians who made at least 1+ Entresto prescriptions a year and did not receive Entresto payments
C = MERGED_DATASET[
    (MERGED_DATASET["ALL_PRESCRIPTIONS"] >= 70) & 
    (MERGED_DATASET["ENTRESTO_PRESCRIPTIONS"] > 0) &
    (MERGED_DATASET["ENTRESTO_PAYMENTS"] == 0) 
    ]

#D: physicians who did not make any Entresto prescriptions and did not receive Entresto payments
D = MERGED_DATASET[
    (MERGED_DATASET["ALL_PRESCRIPTIONS"] >= 70) & 
    (MERGED_DATASET["ENTRESTO_PRESCRIPTIONS"] == 0) &
    (MERGED_DATASET["ENTRESTO_PAYMENTS"] == 0)
    ]

In [28]:
A_size = A['Covered_Recipient_NPI'].nunique()
B_size = B['Covered_Recipient_NPI'].nunique()
C_size = C['Covered_Recipient_NPI'].nunique()
D_size = D['Covered_Recipient_NPI'].nunique()

For every $100$ doctors who got a meal payment, $33$ of them will write an Entresto prescription:

In [29]:
(A_size / B_size) * 100

33.02385065107627

For every $100$ doctors who did NOT get a meal payment, $11$ of them will write an Entresto prescription:

In [30]:
(C_size / D_size) * 100

11.348726817007512

Therefore, the _relative likelihood_ (or relative risk) that a doctor would write a prescription for Entresto if they have received a meal payment versus if they have not is:

In [31]:
RR = (A_size / B_size) / (C_size / D_size)
f'{RR:.2f}'

'2.91'

In [39]:
MERGED_DATASET.to_parquet('../data/compiled_dataset.parquet')